# Mnemosis 一分钟体验

像人脑一样记事的 AI 记忆层：会记住、会遗忘、会整理、会自我怀疑。

在 Colab 里直接运行下面所有单元格即可，无需注册、无需 API key。

In [ ]:
!pip install -q mnemosis

In [ ]:
from mnemosis import MemoryEngine
from mnemosis.types import MemoryKind, SourceRecord, SourceType

engine = MemoryEngine()  # 传一个路径（如 "memory.db"）即可持久化
user = SourceRecord(origin=SourceType.USER)

engine.remember("用户喜欢用中文讨论技术问题。", kind=MemoryKind.SEMANTIC, source=user, cues=["语言", "偏好"])
engine.remember("昨天一起修了 SQLite 锁死的问题。", kind=MemoryKind.EPISODIC, source=user, cues=["SQLite"])

In [ ]:
for r in engine.recall("用户用什么语言聊天？", top_k=3):
    print(f"[{r.item.kind.value}] 相关度 {r.score:.2f}  {r.item.content}")

In [ ]:
# 新旧矛盾：新的价格应该赢
engine.remember("2026年7月1日 物业费调整为一年3000元。", kind=MemoryKind.EPISODIC, source=user, cues=["2026-07-01", "物业费"])
engine.remember("2026年1月6日 缴纳物业费一年2400元。", kind=MemoryKind.EPISODIC, source=user, cues=["2026-01-06", "物业费"])
rows = [r.item.content for r in engine.recall("现在物业费一年多少钱？", top_k=2)]
print("检索到：", " / ".join(rows))

In [ ]:
# 睡眠整合：离线去重、提炼、查矛盾
engine.remember("用户喜欢用中文讨论技术问题。", kind=MemoryKind.SEMANTIC, source=user, cues=["语言", "偏好"])
print(engine.sleep().summary())
print("活跃记忆数：", engine.stats()["active"])

In [ ]:
# 元认知：没把握就直说
check = engine.check("用户最喜欢的电影是什么？")
print("知识缺口：", check.gaps or "无")
print("矛盾条数：", len(check.contradictions))

## 更多

- 中文文档与功能列表：[GitHub 仓库](https://github.com/)（README）
- MCP 接入：Claude Desktop / Cursor / Codex 一行配置，见仓库 `docs/mcp-quickstart.md`
- 命令行：`mnemosis remember ...`、`mnemosis recall ...`